# ASL Alphabet Recognition — EfficientNetV2-S (Preprocessing + Training + Validation)

This notebook implements the full pipeline for training an **EfficientNetV2-S**-based ASL alphabet recognition model.

### Why EfficientNetV2-S over ResNet50 / MobileNetV2?
| Property | MobileNetV2 | ResNet50 | **EfficientNetV2-S** |
|---|---|---|---|
| Input size | 128 × 128 | 224 × 224 | **300 × 300** (recommended) or 224 × 224 |
| Params | 3.4 M | 25 M | **24 M** |
| Top-1 ImageNet | 71.8% | 76.0% | **83.9%** |
| Inference speed | ~120 fps | ~55 fps | **~85 fps** |
| Training speed | Fast | Moderate | **Fast (progressive learning)** |
| Normalization | `[-1, 1]` | ImageNet mean sub | **`[0, 1]` + EfficientNet preprocess** |

EfficientNetV2-S uses **Fused-MBConv** blocks in early stages and **MBConv** blocks in later stages, combining the high accuracy of large models with training efficiency comparable to lightweight ones. It achieves better accuracy than ResNet50 while using similar VRAM.

### Notebook structure
1. Imports & global config  
2. Class mapping  
3. EfficientNetV2-S–compatible preprocessing  
4. Dataset loading & stratified splitting  
5. `tf.data` pipeline with augmentation  
6. Model construction (feature extraction → fine-tuning)  
7. Training — Phase 1 (frozen backbone)  
8. Fine-tuning — Phase 2 (partial unfreeze)  
9. Evaluation & visualisation  
10. Artefact export  

## 1 · Imports & Global Configuration

In [ ]:
import os
import json
import math
import random
from pathlib import Path
from typing import Tuple, List

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as efficientnet_preprocess
from tensorflow.keras.utils import to_categorical

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPUs available    :", tf.config.list_physical_devices("GPU"))

In [ ]:
# ── EfficientNetV2-S Image Requirements ───────────────────────────────────────
# EfficientNetV2-S natively accepts 300×300 RGB for best accuracy.
# 224×224 also works and saves ~30% VRAM — set USE_NATIVE_SIZE = True for 300px.
USE_NATIVE_SIZE = True
IMAGE_SIZE      = 300 if USE_NATIVE_SIZE else 224
NUM_CHANNELS    = 3          # RGB
RANDOM_STATE    = SEED

# ── Training Hyperparameters ─────────────────────────────────────────────────
BATCH_SIZE    = 32           # fits comfortably in 8 GB VRAM at 300×300
EPOCHS_PHASE1 = 15           # frozen backbone — train head only
EPOCHS_PHASE2 = 25           # fine-tune top blocks
LR_PHASE1     = 1e-3
LR_PHASE2     = 5e-5         # EfficientNet fine-tuning is sensitive; keep LR low
DROPOUT_RATE  = 0.35
L2_REG        = 1e-4

# EfficientNetV2-S has 369 layers total.
# Unfreeze the last two blocks (block6 onward) for Phase 2 fine-tuning.
# block5 ends around layer 339 — unfreeze from there.
UNFREEZE_FROM = 340

# ── Paths ────────────────────────────────────────────────────────────────────
RAW_DATA_DIR       = Path("../data/raw/train")
PROCESSED_DATA_DIR = Path("../data/processed_efficientnet")
MODEL_DIR          = Path("../models")
REPORTS_DIR        = Path("../reports")

for d in [PROCESSED_DATA_DIR, MODEL_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Image size : {IMAGE_SIZE}×{IMAGE_SIZE} (native={USE_NATIVE_SIZE})")
print("Directories ready.")

## 2 · Class Mapping

In [ ]:
class_names = sorted([
    d.name for d in RAW_DATA_DIR.iterdir()
    if d.is_dir()
])

num_classes    = len(class_names)
class_to_index = {cls: idx for idx, cls in enumerate(class_names)}
index_to_class = {idx: cls for cls, idx in class_to_index.items()}

print(f"Detected {num_classes} classes:")
print(class_names)

# Persist mappings
with open(PROCESSED_DATA_DIR / "class_mapping.json", "w") as f:
    json.dump({
        "class_to_index" : class_to_index,
        "index_to_class" : {str(k): v for k, v in index_to_class.items()}
    }, f, indent=4)
print("Class mapping saved.")

## 3 · EfficientNetV2-S–Compatible Preprocessing

EfficientNetV2-S uses a **different normalisation scheme** from both MobileNetV2 and ResNet50:

| Model | Preprocessing |
|---|---|
| MobileNetV2 | `img / 127.5 - 1`  →  `[-1, 1]` |
| ResNet50 | ImageNet channel-wise mean subtraction |
| **EfficientNetV2-S** | `img / 255.0`  →  `[0, 1]`  (via `efficientnet_v2.preprocess_input`) |

`efficientnet_v2.preprocess_input` simply rescales pixels to `[0, 1]` — no mean subtraction. This is intentional: EfficientNetV2 includes **batch normalisation** in every block that handles the rest of the normalisation internally. Importantly, **do not apply any other normalisation** before calling `preprocess_input` or you will double-normalise.

In [ ]:
def preprocess_image(image_path: Path) -> np.ndarray:
    """
    Load and preprocess a single image for EfficientNetV2-S.

    Steps
    -----
    1. Load as BGR (OpenCV default)
    2. Convert BGR → RGB
    3. Resize to IMAGE_SIZE × IMAGE_SIZE (bilinear)
    4. Cast to float32 — keep range [0, 255]
    5. Apply efficientnet_v2.preprocess_input  (rescales to [0, 1])

    Returns
    -------
    np.ndarray of shape (IMAGE_SIZE, IMAGE_SIZE, 3), dtype float32, range [0, 1]

    Notes
    -----
    Do NOT manually divide by 255 before calling preprocess_input —
    that function handles scaling internally.
    """
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f"Failed to read image: {image_path}")

    # BGR → RGB (EfficientNetV2 expects RGB)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Resize to target resolution
    img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)

    # float32 in [0, 255] — preprocess_input rescales to [0, 1] internally
    img = img.astype(np.float32)

    # EfficientNetV2 normalisation: [0, 255] → [0, 1]
    img = efficientnet_preprocess(img)

    return img

### Preprocessing sanity-check

In [ ]:
# Verify pixel range after preprocessing
sample_path = next(next(RAW_DATA_DIR.iterdir()).iterdir())
sample_processed = preprocess_image(sample_path)
print(f"Shape after preprocessing : {sample_processed.shape}")
print(f"Pixel range               : [{sample_processed.min():.4f}, {sample_processed.max():.4f}]")
print(f"dtype                     : {sample_processed.dtype}")
assert sample_processed.min() >= 0.0, "Min pixel should be >= 0.0 for EfficientNetV2"
assert sample_processed.max() <= 1.0, "Max pixel should be <= 1.0 for EfficientNetV2"
print("Preprocessing range check passed.")

In [ ]:
# Visual sanity-check — first 10 classes (displayed without un-normalising since [0,1] renders fine)
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for ax, cls in zip(axes, class_names[:10]):
    sample = next((RAW_DATA_DIR / cls).iterdir())
    img_processed = preprocess_image(sample)      # already [0, 1]
    ax.imshow(img_processed)
    ax.set_title(cls, fontsize=12)
    ax.axis("off")

plt.suptitle(f"Sample images — RGB {IMAGE_SIZE}×{IMAGE_SIZE}, EfficientNetV2 normalised [0, 1]",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "sample_images_efficientnet.png", dpi=150)
plt.show()

## 4 · Dataset Loading, Splitting & Saving

In [ ]:
X: List[np.ndarray] = []
y: List[int] = []

for class_name in class_names:
    class_dir = RAW_DATA_DIR / class_name
    label = class_to_index[class_name]

    for img_file in class_dir.iterdir():
        if img_file.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        try:
            img = preprocess_image(img_file)
            X.append(img)
            y.append(label)
        except Exception as e:
            print(f"Skipping {img_file}: {e}")

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

print(f"Dataset loaded — X: {X.shape}, y: {y.shape}")
print(f"Pixel range after preprocess_input : [{X.min():.4f}, {X.max():.4f}]")
print(f"Memory usage : {X.nbytes / 1e9:.2f} GB")

In [ ]:
# ── Stratified 80/20 split ────────────────────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

y_train_oh = to_categorical(y_train, num_classes)
y_val_oh   = to_categorical(y_val,   num_classes)

print("Training   :", X_train.shape, y_train_oh.shape)
print("Validation :", X_val.shape,   y_val_oh.shape)

In [ ]:
# ── Persist preprocessed arrays ───────────────────────────────────────────────
np.save(PROCESSED_DATA_DIR / "X_train.npy", X_train)
np.save(PROCESSED_DATA_DIR / "X_val.npy",   X_val)
np.save(PROCESSED_DATA_DIR / "y_train.npy", y_train_oh)
np.save(PROCESSED_DATA_DIR / "y_val.npy",   y_val_oh)
print("Preprocessed arrays saved to", PROCESSED_DATA_DIR)

## 5 · `tf.data` Input Pipeline

EfficientNetV2-S was pretrained with **progressive learning** — images were shown at increasing resolutions during training. To benefit from this during fine-tuning, augmentations that preserve spatial structure (zoom, rotation, translation) are more important than aggressive colour jitter.

In [ ]:
# ── Augmentation layer (training only) ───────────────────────────────────────
# EfficientNetV2 is particularly responsive to spatial augmentations.
# Keep colour augmentations mild to avoid conflicting with ImageNet pretraining.
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),           # ±36° — ASL hand poses tolerate rotation
    layers.RandomZoom(0.12),
    layers.RandomTranslation(0.06, 0.06),
    layers.RandomBrightness(0.10),
    layers.RandomContrast(0.10),
], name="augmentation")

AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(X: np.ndarray, y: np.ndarray,
                 batch_size: int, augment: bool = False) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if augment:
        ds = ds.shuffle(buffer_size=len(X), seed=SEED)
        ds = ds.map(
            lambda x, lbl: (data_augmentation(x, training=True), lbl),
            num_parallel_calls=AUTOTUNE
        )
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(X_train, y_train_oh, BATCH_SIZE, augment=True)
val_ds   = make_dataset(X_val,   y_val_oh,   BATCH_SIZE, augment=False)

print(f"Train batches : {len(train_ds)}")
print(f"Val batches   : {len(val_ds)}")

## 6 · Model Construction

Strategy:
- Load `EfficientNetV2S(weights='imagenet', include_top=False)` as the backbone
- **Phase 1** — freeze all backbone layers; train only the classification head
- **Phase 2** — unfreeze from block6 onward (deeper Fused-MBConv blocks); fine-tune with a very low LR

**Important:** EfficientNetV2-S contains `BatchNormalization` layers throughout. These are kept **frozen** during fine-tuning to prevent the running statistics from drifting away from ImageNet pretraining values, which would degrade performance.

In [ ]:
def build_model(num_classes: int,
                dropout_rate: float = DROPOUT_RATE,
                l2_reg: float = L2_REG) -> tuple:
    """
    EfficientNetV2-S transfer-learning model for ASL classification.

    Architecture
    ------------
    Input (IMAGE_SIZE, IMAGE_SIZE, 3)
      └─ EfficientNetV2S backbone (frozen initially)
         └─ GlobalAveragePooling2D
            └─ BatchNormalization
               └─ Dense(512, swish, L2)     ← swish matches EfficientNet's internal activation
                  └─ Dropout
                     └─ Dense(256, swish, L2)
                        └─ Dropout
                           └─ Dense(num_classes, softmax)

    Returns
    -------
    (model, backbone) — backbone returned separately so Phase 2 can unfreeze it.
    """
    inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, NUM_CHANNELS), name="input")

    backbone = EfficientNetV2S(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs
    )
    backbone.trainable = False   # Phase 1: frozen

    x = backbone.output
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.BatchNormalization(name="bn_head")(x)

    # Swish activation mirrors EfficientNet's internal design
    x = layers.Dense(512, activation="swish",
                     kernel_regularizer=regularizers.l2(l2_reg),
                     name="dense_512")(x)
    x = layers.Dropout(dropout_rate, name="drop_1")(x)

    x = layers.Dense(256, activation="swish",
                     kernel_regularizer=regularizers.l2(l2_reg),
                     name="dense_256")(x)
    x = layers.Dropout(dropout_rate, name="drop_2")(x)

    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    model = keras.Model(inputs, outputs, name="ASL_EfficientNetV2S")
    return model, backbone

model, backbone = build_model(num_classes)

print(f"Total layers in backbone : {len(backbone.layers)}")
print(f"Trainable params (Phase 1): "
      f"{sum(np.prod(v.shape) for v in model.trainable_variables):,}")
model.summary(show_trainable=True)

## 7 · Phase 1 — Feature Extraction (Frozen Backbone)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_PHASE1),
    loss="categorical_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc")
    ]
)

cb_phase1 = [
    callbacks.ModelCheckpoint(
        filepath=str(MODEL_DIR / "efficientnetv2s_asl_phase1_best.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    callbacks.TensorBoard(
        log_dir=str(REPORTS_DIR / "tb_logs" / "efficientnet_phase1"),
        histogram_freq=1
    )
]

print(f"Phase 1: training head only — {EPOCHS_PHASE1} epochs max")
history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=cb_phase1,
    verbose=1
)

### Phase 1 Learning Curves

In [ ]:
def plot_history(history, title=""):
    h = history.history
    epochs = range(1, len(h["loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    axes[0].plot(epochs, h["loss"],     label="Train Loss",  linewidth=2)
    axes[0].plot(epochs, h["val_loss"], label="Val Loss",    linewidth=2, linestyle="--")
    axes[0].set_title(f"{title} — Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Accuracy
    axes[1].plot(epochs, h["accuracy"],     label="Train Acc",      linewidth=2)
    axes[1].plot(epochs, h["val_accuracy"], label="Val Acc",        linewidth=2, linestyle="--")
    if "top3_acc" in h:
        axes[1].plot(epochs, h["val_top3_acc"], label="Val Top-3 Acc",
                     linewidth=2, linestyle=":", color="green")
    axes[1].set_title(f"{title} — Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    safe_name = title.lower().replace(" ", "_").replace("—", "").replace("/", "_")
    plt.savefig(REPORTS_DIR / f"curves_{safe_name}.png", dpi=150)
    plt.show()

plot_history(history_p1, "Phase 1 — Feature Extraction")

## 8 · Phase 2 — Fine-Tuning (Partial Backbone Unfreeze)

EfficientNetV2-S is divided into blocks:
- **Block 1–3** — Fused-MBConv (early, low-level features — keep frozen)
- **Block 4–6** — MBConv (high-level features — unfreeze block 6)

We unfreeze from layer index `UNFREEZE_FROM` (≈ start of block 6) and train with LR = `5e-5`. All `BatchNormalization` layers remain frozen to preserve running statistics from ImageNet pretraining.

In [ ]:
# ── Unfreeze top EfficientNetV2-S block ───────────────────────────────────────
backbone.trainable = True

# Freeze all layers before UNFREEZE_FROM
for layer in backbone.layers[:UNFREEZE_FROM]:
    layer.trainable = False

# Always keep BatchNorm frozen — critical for EfficientNet
for layer in backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(1 for l in backbone.layers if l.trainable)
total_count     = len(backbone.layers)
print(f"Trainable backbone layers : {trainable_count} / {total_count}")
print(f"Trainable params (Phase 2): "
      f"{sum(np.prod(v.shape) for v in model.trainable_variables):,}")

# Recompile with a much lower LR — EfficientNet fine-tuning is sensitive
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_PHASE2),
    loss="categorical_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc")
    ]
)

cb_phase2 = [
    callbacks.ModelCheckpoint(
        filepath=str(MODEL_DIR / "efficientnetv2s_asl_phase2_best.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-8,
        verbose=1
    ),
    callbacks.TensorBoard(
        log_dir=str(REPORTS_DIR / "tb_logs" / "efficientnet_phase2"),
        histogram_freq=1
    )
]

print(f"Phase 2: fine-tuning top block — {EPOCHS_PHASE2} epochs max")
history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    callbacks=cb_phase2,
    verbose=1
)

In [ ]:
plot_history(history_p2, "Phase 2 — Fine-Tuning")

## 9 · Evaluation & Visualisation

In [ ]:
# ── Load best Phase 2 checkpoint ──────────────────────────────────────────────
best_model = keras.models.load_model(
    MODEL_DIR / "efficientnetv2s_asl_phase2_best.keras"
)

val_loss, val_acc, val_top3 = best_model.evaluate(val_ds, verbose=0)
print(f"Validation Loss     : {val_loss:.4f}")
print(f"Validation Accuracy : {val_acc*100:.2f}%")
print(f"Validation Top-3    : {val_top3*100:.2f}%")

In [ ]:
# ── Predictions on full validation set ───────────────────────────────────────
y_pred_probs = best_model.predict(val_ds, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_val_oh, axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    cm_norm,
    annot=True, fmt=".2f",
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="Blues", linewidths=0.3,
    ax=ax, annot_kws={"size": 7}
)
ax.set_title("Normalised Confusion Matrix — EfficientNetV2-S ASL", fontsize=14, pad=14)
ax.set_xlabel("Predicted Label", fontsize=11)
ax.set_ylabel("True Label", fontsize=11)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "confusion_matrix_efficientnetv2s.png", dpi=150)
plt.show()

In [ ]:
# ── Per-class accuracy bar chart ─────────────────────────────────────────────
per_class_acc = cm_norm.diagonal()
sorted_idx    = np.argsort(per_class_acc)

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(
    [class_names[i] for i in sorted_idx],
    per_class_acc[sorted_idx],
    color=plt.cm.RdYlGn(per_class_acc[sorted_idx])
)
ax.set_xlabel("Accuracy", fontsize=11)
ax.set_title("Per-class Validation Accuracy — EfficientNetV2-S", fontsize=13)
ax.axvline(x=per_class_acc.mean(), color="navy", linestyle="--",
           label=f"Mean: {per_class_acc.mean():.3f}")
ax.legend()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "per_class_accuracy_efficientnetv2s.png", dpi=150)
plt.show()

In [ ]:
# ── Prediction samples grid ───────────────────────────────────────────────────
# EfficientNetV2 preprocess_input maps [0,255] → [0,1], so images are display-ready as-is.
num_samples = 20
indices     = np.random.choice(len(X_val), num_samples, replace=False)

fig, axes = plt.subplots(4, 5, figsize=(18, 14))
axes = axes.flatten()

for ax, idx in zip(axes, indices):
    img_disp = X_val[idx]                  # already [0, 1], display-ready
    true_lbl = index_to_class[y_val[idx]]
    pred_lbl = index_to_class[y_pred[idx]]
    correct  = (true_lbl == pred_lbl)
    conf     = y_pred_probs[idx, y_pred[idx]]

    ax.imshow(np.clip(img_disp, 0, 1))
    color = "green" if correct else "red"
    ax.set_title(
        f"True: {true_lbl}\nPred: {pred_lbl} ({conf*100:.1f}%)",
        color=color, fontsize=8, fontweight="bold"
    )
    ax.axis("off")
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(3)

plt.suptitle("Prediction Samples with Confidence (green=correct, red=wrong)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "prediction_samples_efficientnetv2s.png", dpi=150)
plt.show()

### Combined training history (Phase 1 + Phase 2)

In [ ]:
def merge_histories(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history.get(key, [])
    return merged

full_history = merge_histories(history_p1, history_p2)
total_epochs = len(full_history["loss"])
p1_end       = len(history_p1.history["loss"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epoch_range = range(1, total_epochs + 1)

for ax, metric, val_metric, ylabel in [
    (axes[0], "loss",     "val_loss",     "Loss"),
    (axes[1], "accuracy", "val_accuracy", "Accuracy")
]:
    ax.plot(epoch_range, full_history[metric],     label="Train",      linewidth=2)
    ax.plot(epoch_range, full_history[val_metric], label="Validation", linewidth=2, linestyle="--")
    ax.axvline(x=p1_end + 0.5, color="gray", linestyle=":", linewidth=1.5, label="Fine-tune start")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(f"Full Training — {ylabel}")
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("EfficientNetV2-S — Combined Phase 1 + Phase 2", fontsize=13)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "full_history_efficientnetv2s.png", dpi=150)
plt.show()

## 10 · Artefact Export

In [ ]:
# ── Save final Keras model ────────────────────────────────────────────────────
final_model_path = MODEL_DIR / "efficientnetv2s_asl_final.keras"
best_model.save(final_model_path)
print(f"Final model saved : {final_model_path}")

In [ ]:
# ── TFLite export for real-time inference ────────────────────────────────────
# Dynamic-range quantisation reduces model size ~4× with minimal accuracy loss.
# For maximum inference speed on CPU/GPU, use float16 quantisation instead.
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]   # dynamic-range quantisation
tflite_model = converter.convert()

tflite_path = MODEL_DIR / "efficientnetv2s_asl.tflite"
tflite_path.write_bytes(tflite_model)
print(f"TFLite model saved : {tflite_path}")
print(f"TFLite size        : {tflite_path.stat().st_size / 1e6:.1f} MB")

In [ ]:
# ── Optional: float16 TFLite (faster GPU inference) ──────────────────────────
converter_f16 = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter_f16.optimizations = [tf.lite.Optimize.DEFAULT]
converter_f16.target_spec.supported_types = [tf.float16]
tflite_f16 = converter_f16.convert()

tflite_f16_path = MODEL_DIR / "efficientnetv2s_asl_f16.tflite"
tflite_f16_path.write_bytes(tflite_f16)
print(f"Float16 TFLite saved : {tflite_f16_path}")
print(f"Float16 TFLite size  : {tflite_f16_path.stat().st_size / 1e6:.1f} MB")

In [ ]:
# ── Persist full training history ─────────────────────────────────────────────
with open(REPORTS_DIR / "training_history_efficientnetv2s.json", "w") as f:
    json.dump({
        k: [float(v) for v in vals]
        for k, vals in full_history.items()
    }, f, indent=4)
print("Training history saved.")

In [ ]:
# ── Training Summary ──────────────────────────────────────────────────────────
summary = {
    "model": "EfficientNetV2-S",
    "pretrained_weights": "ImageNet",
    "num_classes": num_classes,
    "class_names": class_names,
    "image_preprocessing": {
        "target_size": [IMAGE_SIZE, IMAGE_SIZE],
        "color_mode": "RGB",
        "channels": NUM_CHANNELS,
        "normalization": "efficientnet_v2.preprocess_input — rescales [0,255] to [0,1]",
        "activation": "swish (matches EfficientNet internal design)"
    },
    "data_split": {
        "train": int(len(X_train)),
        "validation": int(len(X_val)),
        "stratified": True,
        "random_state": RANDOM_STATE
    },
    "training": {
        "phase1_epochs_run": len(history_p1.history["loss"]),
        "phase1_lr": LR_PHASE1,
        "phase1_frozen_backbone": True,
        "phase2_epochs_run": len(history_p2.history["loss"]),
        "phase2_lr": LR_PHASE2,
        "phase2_unfreeze_from_layer": UNFREEZE_FROM,
        "phase2_batchnorm_frozen": True,
        "batch_size": BATCH_SIZE,
        "dropout_rate": DROPOUT_RATE
    },
    "final_validation": {
        "loss": round(float(val_loss), 4),
        "accuracy": round(float(val_acc), 4),
        "top3_accuracy": round(float(val_top3), 4)
    },
    "artifacts": {
        "model_keras":        str(final_model_path),
        "model_tflite":       str(tflite_path),
        "model_tflite_f16":   str(tflite_f16_path),
        "class_mapping":      str(PROCESSED_DATA_DIR / "class_mapping.json")
    }
}

with open(REPORTS_DIR / "training_summary_efficientnetv2s.json", "w") as f:
    json.dump(summary, f, indent=4)

print("\n" + "="*60)
print("  ASL EfficientNetV2-S Training Complete")
print("="*60)
print(f"  Val Accuracy  : {val_acc*100:.2f}%")
print(f"  Val Top-3     : {val_top3*100:.2f}%")
print(f"  Val Loss      : {val_loss:.4f}")
print(f"  TFLite size   : {tflite_path.stat().st_size / 1e6:.1f} MB")
print(f"  F16 TFLite    : {tflite_f16_path.stat().st_size / 1e6:.1f} MB")
print("="*60)